# COMP447 Colab First Run

**Goal:** Get ECT running on a Colab T4, verify environment, do a dry-run, and launch the first sanity training.

**Runtime:** Must be **T4 GPU** (Runtime > Change runtime type > T4 GPU in browser Colab, or select T4 when connecting from VS Code Colab extension).

**How to use with AI assistance:** Run each cell, wait for output. If a cell fails, the error is saved into this `.ipynb` file and Claude can read it directly from the repo. No copy-paste needed.

---

## Cell 1: Verify GPU

**Expected output:**
```
torch: 2.x.x+cu...
cuda available: True
gpu: Tesla T4
```

If `cuda available: False` → runtime is not T4. Change runtime type and reconnect.
If gpu is not T4 (e.g. `L4`, `A100`) → still OK for dry run, but final latency numbers must come from T4.

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram gb:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## Cell 2: Mount Drive (for persistent outputs)

**Expected output:** `Mounted at /content/drive`

Why mount Drive: Colab sessions reset, but we want checkpoints and results to survive. Drive is the persistent layer.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/COMP447_runs

## Cell 3: Clone the COMP447 wrapper repo

**Expected output:** listing showing `project`, `final_upload`, `readings`, `proposal_template`, `README.md`.

Note: this is only the wrapper repo. ECT and EDM are cloned fresh in the next cell by `setup_ect.sh`. That's why this clone is fast.

In [ ]:
%cd /content
!rm -rf COMP447
!git clone https://github.com/bakaraman/COMP447.git
%cd /content/COMP447
!ls

## Cell 4: Clone ECT and EDM upstream repos

**Expected output:** two `Cloning into ...` messages, then `Bootstrap complete.` and the "Next steps" block.

This runs `project/scripts/setup_ect.sh` which clones:
- `https://github.com/locuslab/ect.git` into `project/src/ect`
- `https://github.com/NVlabs/edm.git` into `project/src/edm`

In [ ]:
%cd /content/COMP447
!bash project/scripts/setup_ect.sh

## Cell 5: Install missing Python packages

**Expected output:** a short pip install log, no errors. Colab already has torch, numpy, scipy, pillow, tqdm preinstalled, so this only fills in the rest.

If `diffusers==0.26.3` or `accelerate==0.27.2` pins conflict with Colab's preinstalled versions, we'll relax them — report back.

In [ ]:
!pip install -q psutil click requests pyspng imageio-ffmpeg diffusers==0.26.3 accelerate==0.27.2

## Cell 6: Prepare CIFAR-10 in EDM format

**Expected output:**
- `cifar-10-python.tar.gz` downloaded (~170 MB)
- `datasets/cifar10-32x32.zip` created (~180 MB)
- Final `ls -lh datasets` shows the zip

`dataset_tool.py` converts the raw CIFAR to the EDM/ECT training format (zipped uint8 images + labels).

In [ ]:
%cd /content/COMP447/project/src/ect
!mkdir -p datasets
!wget -nc -q --show-progress -O /content/cifar-10-python.tar.gz https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz
!python3 dataset_tool.py --source=/content/cifar-10-python.tar.gz --dest=datasets/cifar10-32x32.zip
!ls -lh datasets

## Cell 7: Import smoke test

**Expected output:** `psutil ok`, `torch ok`, `cuda: True`, no ImportError.

This verifies that the packages ECT needs at import time are all present in the current kernel.

In [ ]:
%cd /content/COMP447/project/src/ect
!python3 - <<'PY'
import psutil, torch, click, PIL, numpy, scipy, tqdm, imageio
import pyspng
print("psutil ok")
print("torch ok", torch.__version__)
print("cuda:", torch.cuda.is_available())
PY

## Cell 8: Dry-run the ECT training command

**Expected output:** ECT loads the pretrained EDM checkpoint, prints a summary of training args, and exits (dry run doesn't actually step). No traceback.

If this fails, the error usually points at one of:
- missing package → fix Cell 5
- CUDA mismatch → wrong runtime
- download of pretrained checkpoint failed → retry

Note: `ct_train.py` may not support `--dry_run` flag directly. If it errors on unknown arg, we drop it and set `--duration=0.001` to make it exit fast instead.

In [ ]:
%cd /content/COMP447/project/src/ect
!python3 ct_train.py \
  --outdir ct-runs-dry \
  --data datasets/cifar10-32x32.zip \
  --cond=0 \
  --arch=ddpmpp \
  --metrics=fid50k_full \
  --transfer=https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-uncond-vp.pkl \
  --duration=0.001 \
  --batch=8 \
  --dry_run || echo "=== if --dry_run is unknown, rerun without it and rely on --duration=0.001 ==="

## Cell 9: First real sanity training run (1 hour)

**Expected output:** ECT begins training, prints tick/kimg logs every ~1-2 minutes. Not expected to finish — this is just to confirm the real loop runs on T4.

**Only run this after Cell 8 succeeds.** This consumes GPU time; stop it with the ⏹ button once you see ~2-3 ticks of healthy output, or let it run for the full hour if you have budget.

After stopping, ask Claude to read the output and confirm it looked healthy before moving on to Phase 2.

In [ ]:
%cd /content/COMP447/project/src/ect
!bash run_ecm_1hour.sh 1 29500 --desc monday_sanity

## Cell 10: Save run artifacts to Drive (optional, run after Cell 9)

Copies whatever ECT wrote into `ct-runs/` into Drive so the next session can resume. Run this **before** disconnecting the runtime.

In [ ]:
!mkdir -p /content/drive/MyDrive/COMP447_runs/monday_sanity
!cp -r /content/COMP447/project/src/ect/ct-runs/* /content/drive/MyDrive/COMP447_runs/monday_sanity/ 2>/dev/null || echo "nothing to copy yet"
!ls -lh /content/drive/MyDrive/COMP447_runs/